# Herding-Strength Calibration Check

Tests the fallback pick-share model (herding x availability x urgency) used in `survivor_optimizer.ipynb`'s equity mode against REAL pick-percentage data for a week you actually have it for -- and this is genuinely how `herding_strength=8.0` (the current default) was derived, from real Week 1 2026 data. Re-run this any week you have real pick-percentage data (survivorgrid, Yahoo, etc.) to check whether 8.0 still holds up, or whether new evidence points somewhere else.

**Why this does two checks, not one.** Fitting a parameter to data and then reporting its error *on that same data* proves nothing -- the minimum of a search is definitionally low-error against whatever it was searched over. So this notebook does:

1. **Full-dataset sweep** -- for a range of `herding_strength` values, how well does each one's guess match the real percentages, across every team? This finds a candidate value, but on its own it's a curve-fit, not evidence.
2. **Held-out validation** -- repeatedly split the teams into two random halves, pick the best-fitting value using only HALF the teams, then check how well that *same* choice does on the OTHER half it never saw during the search. If one value keeps winning on data it wasn't fit to, across many random splits, that's real evidence the pattern is genuine rather than an artifact of searching too hard against one dataset.

**Real pick-percentage CSV format** (same as `survivor_optimizer.ipynb`'s `PICKS_CSV`, but for exactly the one week you're calibrating against):
```
week,team,pick_pct
1,LAC,0.276
```

In [11]:
import random
from collections import Counter

import pandas as pd

## Helper Functions

Extracted from `survivor_optimizer.ipynb` so this notebook can run standalone.

In [12]:
def american_to_prob(ml: float) -> float:
    ml = float(ml)
    if ml > 0:
        return 100.0 / (ml + 100.0)
    return -ml / (-ml + 100.0)


def load_odds(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["team"] = df["team"].astype(str).str.upper().str.strip()
    df["opponent"] = df["opponent"].astype(str).str.upper().str.strip()
    df["raw_prob"] = df["moneyline"].apply(american_to_prob)
    df["game_key"] = list(zip(
        df["week"],
        [tuple(sorted(pair)) for pair in zip(df["team"], df["opponent"])],
    ))
    game_sums = df.groupby("game_key")["raw_prob"].transform("sum")
    df["win_prob"] = df["raw_prob"] / game_sums
    return df.drop(columns="game_key")


def load_real_picks(path: str) -> dict:
    """team -> real proj_pick_dist, for ONE week (see CSV format in the config cell)."""
    df = pd.read_csv(path)
    df["team"] = df["team"].astype(str).str.upper().str.strip()
    if df["proj_pick_dist"].max() > 1:
        df["proj_pick_dist"] = df["proj_pick_dist"] / 100.0
    return dict(zip(df["team"], df["proj_pick_dist"]))


def estimate_pick_shares(odds_df, herding_strength):
    """Same fallback model as survivor_optimizer.ipynb -- no real proj_pick_dist
    data at all here, since the whole point is testing the model's OWN
    blind guess against real data, not blending the two."""
    team_week_probs = {}
    for row in odds_df.itertuples():
        team_week_probs.setdefault(row.team, {})[row.week] = row.win_prob

    current_avail = {t: 1.0 for t in odds_df["team"].unique()}
    rows = []
    for week in sorted(odds_df["week"].unique()):
        teams = odds_df[odds_df["week"] == week][["team", "win_prob"]].drop_duplicates("team")
        raw_weights = {}
        for _, r in teams.iterrows():
            t, p = r["team"], r["win_prob"]
            avail = current_avail.get(t, 1.0)
            other = [v for w2, v in team_week_probs[t].items() if w2 != week]
            urgency = p / (p + sum(other) / len(other)) if other else 1.0
            raw_weights[t] = avail * (p ** herding_strength) * urgency
        total = sum(raw_weights.values())
        shares = {t: w / total for t, w in raw_weights.items()} if total > 0 else {}
        for t in teams["team"]:
            rows.append({"week": week, "team": t, "pick_share": shares.get(t, 0.0)})
            current_avail[t] = max(0.0, current_avail.get(t, 1.0) - shares.get(t, 0.0))
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# The actual calibration check

## Calibration Check

In [13]:
def run_calibration(odds_df, real_picks, week, h_range, n_trials=50, seed=42):
    teams = list(real_picks.keys())

    precomputed = {}
    for h in h_range:
        shares = estimate_pick_shares(odds_df, herding_strength=h)
        precomputed[h] = shares[shares.week == week].set_index("team")["pick_share"]

    def mae(h, subset):
        guessed = precomputed[h]
        return sum(abs(guessed[t] - real_picks[t]) for t in subset) / len(subset)

    print(f"=== Full-dataset sweep (week {week}, {len(teams)} teams) ===")
    print(f"{'h':<6}{'mean abs error':<16}")
    full_mae = {}
    for h in h_range:
        full_mae[h] = mae(h, teams)
        print(f"{h:<6}{full_mae[h]*100:<16.3f}")
    best_full_h = min(full_mae, key=full_mae.get)
    print(f"\nBest fit on full dataset: h={best_full_h} ({full_mae[best_full_h]*100:.3f}pp error)")
    print("(this alone is NOT proof -- it's the minimum of a search against")
    print(" the same data being scored, which is guaranteed to look good)")

    print(f"\n=== Held-out validation: {n_trials} random train/test splits ===")
    random.seed(seed)
    chosen_counts = Counter()
    ratios = []
    half = len(teams) // 2
    for _ in range(n_trials):
        shuffled = teams[:]
        random.shuffle(shuffled)
        train, test = shuffled[:half], shuffled[half:]
        best_h = min(h_range, key=lambda h: mae(h, train))
        train_err = mae(best_h, train)
        test_err = mae(best_h, test)
        chosen_counts[best_h] += 1
        if train_err > 0:
            ratios.append(test_err / train_err)

    print(f"\nDistribution of h chosen across {n_trials} splits (fit on half the teams, never seeing the other half):")
    for h in sorted(chosen_counts):
        pct = chosen_counts[h] / n_trials * 100
        print(f"  h={h:<4} chosen {chosen_counts[h]:3d} times  ({pct:.0f}%)")

    ratios_sorted = sorted(ratios)
    median_ratio = ratios_sorted[len(ratios_sorted)//2]
    print(f"\nHeld-out/train error ratio: median={median_ratio:.2f}x   worst={max(ratios):.2f}x   best={min(ratios):.2f}x")
    print("(near 1.0 = the chosen h generalizes well to teams it never saw;")
    print(" far above 1.0 on a given split = that split's choice was overfit)")

    most_common_h, most_common_count = chosen_counts.most_common(1)[0]
    agreement = most_common_count / n_trials * 100
    print(f"\n=== Conclusion ===")
    print(f"h={most_common_h} was chosen on {agreement:.0f}% of held-out splits -- this, not the full-dataset")
    print(f"fit alone, is the actual evidence for what herding_strength should be.")

    return full_mae, chosen_counts, ratios

## Config

- `WEEK`: which week's real data you're calibrating against.
- `H_RANGE`: the range of `herding_strength` values to sweep.
- `N_TRIALS`: how many random train/test splits to run for the held-out check -- 50 is a reasonable default, higher gives a smoother distribution at the cost of runtime (each trial re-scores every candidate h, but reuses the same precomputed guesses, so it's cheap).

In [14]:
ODDS_CSV = "odds.csv"           # your full-season odds CSV
REAL_PICKS_CSV = "projected_pick_distribution.csv"  # real pick_pct data for ONE week
WEEK = 2                                  # which week the real data above is for
H_RANGE = range(2, 15)                     # herding_strength values to test
N_TRIALS = 50                              # number of random train/test splits
SEED = 42                                  # for reproducibility -- change to sanity-check stability

## Run

In [15]:
odds_df = load_odds(ODDS_CSV)
real_picks = load_real_picks(REAL_PICKS_CSV)

run_calibration(odds_df, real_picks, WEEK, H_RANGE, N_TRIALS, SEED)

=== Full-dataset sweep (week 2, 32 teams) ===
h     mean abs error  
2     3.084           
3     2.739           
4     2.448           
5     2.227           
6     2.024           
7     1.834           
8     1.692           
9     1.752           
10    1.816           
11    1.894           
12    2.003           
13    2.110           
14    2.218           

Best fit on full dataset: h=8 (1.692pp error)
(this alone is NOT proof -- it's the minimum of a search against
 the same data being scored, which is guaranteed to look good)

=== Held-out validation: 50 random train/test splits ===

Distribution of h chosen across 50 splits (fit on half the teams, never seeing the other half):
  h=8    chosen  26 times  (52%)
  h=10   chosen   1 times  (2%)
  h=11   chosen   3 times  (6%)
  h=13   chosen   3 times  (6%)
  h=14   chosen  17 times  (34%)

Held-out/train error ratio: median=1.74x   worst=14.13x   best=0.27x
(near 1.0 = the chosen h generalizes well to teams it never saw;
 far 

({2: np.float64(0.030843139642519054),
  3: np.float64(0.027386151274119704),
  4: np.float64(0.024475596624774425),
  5: np.float64(0.022267171835973366),
  6: np.float64(0.020244702155540452),
  7: np.float64(0.018343200253144707),
  8: np.float64(0.016916036024101175),
  9: np.float64(0.01752262742314025),
  10: np.float64(0.018155977725397825),
  11: np.float64(0.01893712589376166),
  12: np.float64(0.020030876998255266),
  13: np.float64(0.021095914027872725),
  14: np.float64(0.022182402544306545)},
 Counter({8: 26, 14: 17, 13: 3, 11: 3, 10: 1}),
 [np.float64(2.051892187120506),
  np.float64(1.230658720537264),
  np.float64(0.27032356403955515),
  np.float64(2.4704874242750687),
  np.float64(0.9677602442098894),
  np.float64(0.7160487618089719),
  np.float64(2.3500777191118747),
  np.float64(0.6845719306837524),
  np.float64(1.7042731651976761),
  np.float64(2.827883658314581),
  np.float64(5.102737055586474),
  np.float64(1.382889772149048),
  np.float64(2.8194351631494956),
  n